In [ ]:
from datetime import datetime
import shutil
import torch
import numpy as np
import rasterio
import os
from rasterio.windows import Window
from time import sleep

# DONE: Made test loader georeference LST
# DONE: Make test loader georeference all the rest to check ground truth against original as tile
# DONE: Test making each tile
# DONE: Test 0.0 overlap
# DONE: Organize to inference script and test the data loading more
# DONE: Make model prediction also pass geo reference and test against original as tile
# DONE: Add Heat index 1-25 into preprocess
# DONE: Make Heat Index show up in test quality
# DONE: Make it show in inference
# DONE: Train a preliminary model with 20 epochs, batch 1
# DONE: Test the model in inference
# DONE: Create a nowcast, 3 month cast and 6 month cast according to some parameter
# DONE: Get batching working -> max out the VRAM
# UNECESSARY: In inference, combine the tiles by city. Make it possible to do partial. Don't combine not in same city
# TODO: Make sure 512 and batching does have proper loss
# TODO: Get stats of all
# TODO: Preprocess all
# TODO: Implement normalization
# TODO: Test proper loss with normalization
# TODO: Look for the augmentation code in Teams or ask Isaac
# TODO: Migrate to Jetstream
# TODO: Train and test with augmentation
# DONE: Create a way to look 0, 3, 6 months ahead
# TODO: Test by zero-shot, city only, temporal only, both
# TODO: Report to Isaac 

In [2]:
def inference(model, test_loader, tiles_count: int, device='cuda'):
    model = model.to(device)
    model.eval()    
    batch = 0
    it = iter(test_loader)
    with torch.no_grad():
        for _ in range(tiles_count):
            sleep(1)
            # Get one sample
            sample = next(it)
            for l, outTif in enumerate(['LST.tif', 'HeatIndex.tif']):
                inputs = sample['input'].to(device)
                targets = sample['target'][batch][l].to(device)
                mask = sample['mask'].to(device)
                ground_truth_file = sample['file_dict'][outTif][0]
                box = sample['box']
                box = [int(tensor.item()) for tensor in box]

                # Get model prediction
                outputs = model(inputs)[batch][l]

                # Move to CPU and convert to numpy
                mask_np = mask.cpu().numpy().squeeze()
                targets_np = targets.cpu().numpy().squeeze()
                predicted_np = outputs.cpu().numpy().squeeze()
                
                # Apply mask
                predicted_np[~mask_np] = np.nan
                targets_np[~mask_np] = np.nan

                output_dir = "./Data/prediction"
                os.makedirs(output_dir, exist_ok=True)
                xmin, ymin, xmax, ymax = box
                window = Window(col_off=xmin, row_off=ymin, width=xmax-xmin, height=ymax-ymin)
                
                # Get corresponding LST file path and save outputs
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                with rasterio.open(ground_truth_file) as src:
                    profile = src.profile.copy()
                    # Update the transform based on the window
                    window_transform = rasterio.windows.transform(window, src.transform)
                    
                    # Update profile with new dimensions and transform
                    profile.update(
                        width=xmax-xmin,
                        height=ymax-ymin,
                        transform=window_transform,
                        count=1,
                        nodata=np.nan if outTif is 'LST.tif' else 0
                    )
                    
                    # Save prediction
                    pred_filename = os.path.join(output_dir, f'predicted_{timestamp}_{outTif}')
                    with rasterio.open(pred_filename, "w", **profile) as dst:
                        dst.write(predicted_np.astype(np.float32), 1)
                    
                    # Save ground truth
                    # truth_filename = os.path.join(output_dir, f'ground_truth_LST_{timestamp}.tif')
                    # with rasterio.open(truth_filename, "w", **profile) as dst:
                    #     dst.write(targets_np.astype(np.float32), 1)
                    
                    # Copy original out file
                    orig_filename = os.path.join(output_dir, f'original_{outTif}')
                    if not os.path.exists(orig_filename):
                        print(f"Original LST: {os.path.basename(orig_filename)}")
                        shutil.copy2(ground_truth_file, orig_filename)
                    
                    # Calculate metrics for valid pixels
                    valid_mask = ~np.isnan(predicted_np)
                    if valid_mask.any():
                        mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                        rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                        metrics = {'mae': mae, 'rmse': rmse}
                        print(f"Predictions: {os.path.basename(pred_filename)}")
                        # print(f"Ground Truth: {os.path.basename(truth_filename)}")
                        if 'Heat' in pred_filename:
                            print(f"Mean Absolute Error: {mae:.2f} points.")
                            print(f"Root Mean Square Error: {rmse:.2f} points.")
                        else:
                            print(f"Mean Absolute Error: {mae:.2f}°F")
                            print(f"Root Mean Square Error: {rmse:.2f}°F")
                print(f"\nSaved files in {output_dir}/:")
        
    # return metrics

def test_data_quality(test_loader, tiles_count: int):
    batch = 0
    it = iter(test_loader)
    for _ in range(tiles_count):
        sleep(1)
        # Get one sample
        sample = next(it)
        for i, tif in enumerate(['Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif', 'LST.tif', 'HeatIndex.tif']):
            with torch.no_grad():
                if i <= 4:
                    targets = sample['input'][batch][i]
                else:
                    targets = sample['target'][batch][i-5]
                mask = sample['mask']
                target_file_origin = sample['file_dict'][tif][0]
                box = sample['box']
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                box = [int(tensor.item()) for tensor in box]

                # Move to CPU and convert to numpy
                mask_np = mask.cpu().numpy().squeeze()
                targets_np = targets.cpu().numpy().squeeze()

                noData = np.nan            
                if tif in ['Albedo.tif', 'NDVI.tif', 'NDWI.tif', 'Land_Cover.tif', 'HeatIndex.tif']:
                    noData = 0
                elif tif in ['LST.tif']:
                    noData = 250
                elif tif in ['DEM.tif']:
                    noData = -32767
                targets_np[~mask_np] = noData
                # Save ground truth
                output_dir = "./Data/truth"
                os.makedirs(output_dir, exist_ok=True)
                xmin, ymin, xmax, ymax = box
                window = Window(col_off=xmin, row_off=ymin, width=xmax-xmin, height=ymax-ymin)
                
                with rasterio.open(target_file_origin) as src:
                    # Get the profile from the source
                    test_profile = src.profile.copy()
                    
                    # Update the transform based on the window
                    window_transform = rasterio.windows.transform(window, src.transform)
                    
                    # Update profile with new dimensions and transform
                    test_profile.update(
                        width=xmax-xmin,
                        height=ymax-ymin,
                        transform=window_transform,
                        count=1,
                        nodata=noData
                    )
                    
                    truth_filename = os.path.join(output_dir, f'ground_truth_{timestamp}_{tif}')
                    with rasterio.open(truth_filename, "w", **test_profile) as dst:
                        dst.write(targets_np.astype(np.float32), 1)

                orig_filename = os.path.join(output_dir, f'original_File_{timestamp}_{tif}')
                with rasterio.open(target_file_origin) as src:
                    profile = src.profile.copy()
                    profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)
                    print(f"\nSaved files in {output_dir}/:")
                    print(f"Ground Truth: {os.path.basename(truth_filename)}")
                    print(f"Original LST: {os.path.basename(orig_filename)}")
                    # Copy original output files
                    print(f'Copying {target_file_origin} to {orig_filename}')
                    if not os.path.exists(orig_filename):
                        shutil.copy2(target_file_origin, orig_filename)

<>:49: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:49: SyntaxWarning: "is" with a literal. Did you mean "=="?
/work/ubh496/podman_storage/ipykernel_2736977/2478896375.py:49: SyntaxWarning: "is" with a literal. Did you mean "=="?
  nodata=np.nan if outTif is 'LST.tif' else 0


In [3]:
# import wandb
# import os

# os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"
# os.environ["WANDB_DIR"] = "./wandb"
# os.environ["WANDB_CACHE_DIR"] = "/work/ubh496/.cache/wandb"
# os.environ["WANDB_CONFIG_DIR"] = "/work/ubh496/.config/wandb"
# os.environ["WANDB_DATA_DIR"] = "/work/ubh496/.cache/wandb-data"
# os.environ["WANDB_ARTIFACT_DIR"] = "./artifacts"

# run = wandb.init(dir="/work/ubh496/heat-island-test/wandb/downloaded_models")
# artifact = run.use_artifact('jesus-guerrero-ml/heat-island/model-sclb910d:v8', type='model')
# artifact_dir = artifact.download()

In [4]:
from utils.data.TiledLandsatDataModule import TiledLandsatDataModule
from utils.model import LSTNowcaster

config = {
    "debug": True,
    "by_city": False,
    "months_ahead": 3,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 25,
    "batch_size": 1,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

best_model = LSTNowcaster.load_from_checkpoint(
    "/work/ubh496/heat-island-test/wandb/heat-island/checkpoints/epoch=13-step=13944.ckpt"
)

data_module = TiledLandsatDataModule(
    data_dir="./Data",
    monthsAhead=config["months_ahead"],
    batch_size=1,
    num_workers=5,        
    tile_size=512,  # Adjust based on your GPU memory
)
data_module.setup()


/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/torch/hub.py:846: UserWarning: TORCH_MODEL_ZOO is deprecated, please use env TORCH_HOME instead
  warnings.warn(
Preparing scene by scene...: 100%|██████████| 650/650 [00:01<00:00, 420.80it/s]


In [5]:
test_data_quality(
    test_loader=data_module.test_dataloader(), tiles_count=3
)

# inference(
#     model=best_model, test_loader=data_module.val_dataloader(), tiles_count=3
# )



Saved files in ./Data/truth/:
Ground Truth: ground_truth_20250309_105326_Albedo.tif
Original LST: original_File_20250309_105326_Albedo.tif
Copying /work/ubh496/heat-island-test/Data/preprocess/X/less5CloudCover/Fayetteville_NC/2014-03/Albedo.tif to ./Data/truth/original_File_20250309_105326_Albedo.tif

Saved files in ./Data/truth/:
Ground Truth: ground_truth_20250309_105326_DEM.tif
Original LST: original_File_20250309_105326_DEM.tif
Copying /work/ubh496/heat-island-test/Data/preprocess/X/less5CloudCover/Fayetteville_NC/2014-03/DEM.tif to ./Data/truth/original_File_20250309_105326_DEM.tif

Saved files in ./Data/truth/:
Ground Truth: ground_truth_20250309_105326_Land_Cover.tif
Original LST: original_File_20250309_105326_Land_Cover.tif
Copying /work/ubh496/heat-island-test/Data/preprocess/X/less5CloudCover/Fayetteville_NC/2014-03/Land_Cover.tif to ./Data/truth/original_File_20250309_105326_Land_Cover.tif

Saved files in ./Data/truth/:
Ground Truth: ground_truth_20250309_105326_NDVI.tif
O